## EN3150 Assignment 03: Resource-Constrained CNN for Image Classification

### Animals-10 Dataset

**Objective:** Develop and evaluate convolutional neural networks for classifying animal images.

The dataset contains ten animal classes. All images will be resized to 64 × 64 pixels and represented using RGB channels.

The dataset will be divided into three subsets:

- Training set: 70%
- Validation set: 15%
- Testing set: 15%

The same dataset split will be used for all models to ensure a fair comparison.

**Framework:** TensorFlow and Keras

**Dataset:** Animals-10

In [8]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv

from pathlib import Path
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("OpenCV version:", cv.__version__)
print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

OpenCV version: 5.0.0
TensorFlow version: 2.22.0-rc0
NumPy version: 2.5.3
GPU devices: []


In [14]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "raw-img"

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATASET_DIR)
print("Dataset directory exists:", DATASET_DIR.exists())

Project directory: e:\SEM 5\pattern\Assignment 03\EN3150_Animal_Classification_CNN
Dataset directory: e:\SEM 5\pattern\Assignment 03\EN3150_Animal_Classification_CNN\raw-img
Dataset directory exists: True


In [16]:
class_folders = sorted(
    folder for folder in DATASET_DIR.iterdir()
    if folder.is_dir()
)

print("Number of classes:", len(class_folders))
print("\nAnimal classes:")

for folder in class_folders:
    print(folder.name)

Number of classes: 10

Animal classes:
butterfly
cat
chicken
cow
dog
elephant
horse
sheep
spider
squirrel


In [17]:
valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_counts = {}

for folder in class_folders:
    count = 0

    for file in folder.iterdir():
        if file.is_file() and file.suffix.lower() in valid_extensions:
            image = cv.imread(str(file))

            if image is not None:
                count += 1

    class_counts[folder.name] = count

class_counts_df = pd.DataFrame(
    list(class_counts.items()),
    columns=["Animal Class", "Number of Images"]
)

display(class_counts_df)

print("Total valid images:", sum(class_counts.values()))

,Animal Class,Number of Images
0,butterfly,2112
1,cat,1668
2,chicken,3098
3,cow,1866
4,dog,4863
5,elephant,1446
6,horse,2623
7,sheep,1820
8,spider,4821
9,squirrel,1862


Total valid images: 26179


## Stratified Dataset Splitting

The dataset contains an unequal number of images in each animal class. To maintain similar class proportions in the training, validation, and testing sets, stratified sampling is used.

The dataset is divided into 70% training, 15% validation, and 15% testing data. The same split will be used for all CNN models to ensure a fair comparison. The original class distribution is retained without removing or duplicating images.

In [18]:
from sklearn.model_selection import train_test_split

# Map class names to consistent integer labels
class_names = sorted(
    folder.name for folder in class_folders
)

class_to_index = {
    name: index for index, name in enumerate(class_names)
}

# Collect valid image paths and their labels
image_paths = []
image_labels = []

for folder in class_folders:
    label = class_to_index[folder.name]

    for file in folder.iterdir():
        if file.is_file() and file.suffix.lower() in valid_extensions:
            image = cv.imread(str(file))

            if image is not None:
                image_paths.append(str(file.resolve()))
                image_labels.append(label)

image_paths = np.array(image_paths)
image_labels = np.array(image_labels)

print("Total valid images:", len(image_paths))
print("Number of classes:", len(class_names))
print("Class mapping:", class_to_index)

Total valid images: 26179
Number of classes: 10
Class mapping: {'butterfly': 0, 'cat': 1, 'chicken': 2, 'cow': 3, 'dog': 4, 'elephant': 5, 'horse': 6, 'sheep': 7, 'spider': 8, 'squirrel': 9}


In [19]:
RANDOM_SEED = 42

X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    image_labels,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=image_labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Test images:", len(X_test))
print("Total:", len(X_train) + len(X_val) + len(X_test))

Training images: 18325
Validation images: 3927
Test images: 3927
Total: 26179


## Image Preprocessing for Model 01

All images are resized to 64 × 64 pixels and converted from BGR to RGB using OpenCV. Pixel values are normalized from the range [0, 255] to [0, 1].

The images are loaded in batches to reduce memory usage during training. The same preprocessing procedure is applied to the training, validation, and test sets.

In [20]:
#Define image preprocessing
IMG_SIZE = 64
BATCH_SIZE = 32
NUM_CLASSES = len(class_names)

def load_image(image_path, label):
    image = cv.imread(image_path)

    if image is None:
        raise ValueError(f"Could not load image: {image_path}")

    image = cv.cvtColor(image, cv.COLOR_BGR2RGB)
    image = cv.resize(
        image,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv.INTER_AREA
    )

    image = image.astype(np.float32) / 255.0

    return image, label

In [21]:
def create_dataset(paths, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels.astype(np.int32))
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(paths), 5000),
            seed=RANDOM_SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        lambda path, label: tf.numpy_function(
            load_image,
            [path, label],
            [tf.float32, tf.int32]
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.map(
        lambda image, label: (
            tf.ensure_shape(image, (IMG_SIZE, IMG_SIZE, 3)),
            tf.ensure_shape(label, ())
        ),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = create_dataset(X_train, y_train, training=True)
val_ds = create_dataset(X_val, y_val)
test_ds = create_dataset(X_test, y_test)

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Testing batches:", tf.data.experimental.cardinality(test_ds).numpy())

Training batches: 573
Validation batches: 123
Testing batches: 123


In [ ]:

images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Minimum pixel value:", tf.reduce_min(images).numpy())
print("Maximum pixel value:", tf.reduce_max(images).numpy())